In [1]:
import os

print("je m'execute depuis :", os.getcwd())
print("le dossier data existe :", os.path.isdir("../data"))
print("le fichier existe     :", os.path.isfile("../data/restaurants.json"))
print("contenu de ../data    :", os.listdir("../data"))

je m'execute depuis : C:\Users\sebas\Documents\json-api-geocodage\notebooks
le dossier data existe : True
le fichier existe     : True
contenu de ../data    : ['carte.html', 'restaurants.json', 'restaurants_geo.json']


In [2]:
import json
with open("../data/restaurants.json", encoding="utf-8") as f:
    data = json.load(f)

In [3]:
print(data.keys())
print(len(data["restaurants"]), "restaurants")
print(data["restaurants"][0])

dict_keys(['ville', 'pays', 'releve', 'restaurants'])
4 restaurants
{'nom': 'Gelateria Zio Pietro dal 1964', 'adresse': 'Via Porta di Ferro, 47, 95131 Catania CT', 'type': 'glacier', 'notes': [4.6, 4.8, 4.7]}


In [4]:
for r in data["restaurants"]:
    r["moyenne"] = round(sum(r["notes"]) / len(r["notes"]), 2) if r["notes"] else None
    print(r["nom"], r["moyenne"])

Gelateria Zio Pietro dal 1964 4.7
A Casa d'Amici 4.45
La Bitta 4.15
Steak House 4.0


In [5]:
import requests

def geocode(adresse):
    rep = requests.get(
        "https://nominatim.openstreetmap.org/search",
        params={"q": adresse, "format": "json", "limit": 1},
        headers={"User-Agent": "quete-json-api"},
        timeout=10,
    )
    rep.raise_for_status()
    res = rep.json()
    return (float(res[0]["lat"]), float(res[0]["lon"])) if res else (None, None)

geocode(data["restaurants"][0]["adresse"])

(37.5020931, 15.0931951)

In [6]:
import time
for r in data["restaurants"]:
    r["lat"], r["lon"] = geocode(r["adresse"])
    print(r["nom"], r["lat"], r["lon"])
    time.sleep(1)  # Nominatim : 1 requete par seconde maximum
assert all(r["lat"] for r in data["restaurants"]), "adresse non trouvee"

Gelateria Zio Pietro dal 1964 37.5020931 15.0931951


A Casa d'Amici 37.506171 15.0943223


La Bitta 37.4894403 15.0794313


Steak House 37.5029335 15.0930406


In [7]:
with open("../data/restaurants_geo.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

In [8]:
import folium
carte = folium.Map(location=[37.502, 15.087], zoom_start=14, tiles="Esri.WorldStreetMap")  # OSM refuse les fichiers locaux (403), CartoDB demande une cle
for r in data["restaurants"]:
    folium.Marker([r["lat"], r["lon"]], popup=f'{r["nom"]} ({r["moyenne"]})').add_to(carte)
carte.save("../data/carte.html")  # ouvrir dans le navigateur si la carte ne s'affiche pas ici
carte